# 03 · HEO stacks and v3 labels from the products

Works on the HEO products in `data/heo/products/`, at the moment the 10 L1C products from August and September 2026. For each product it writes

- a 7-band stack to `data/heo/images/<product>_img.tif`: blue, green, red, RE1, RE2, RE3 and NIR (product bands B01 to B07, PAN left out), uint16 values as delivered, on the grid of B01
- a v3 mask to `data/heo/masks_v3/<product>_mask.tif`

The burn rules are the same as in notebook 02 and come from `burnscar/labels.py`: fresh burn 0 to 90 days, old burn 90 to 365 days, the 10 m edge band and the green or bright burn pixels as ignore. The difference is that there is no v2 mask to start from, so the rest is made here. Clouds and shadows come from OmniCloudMask on red, green and NIR at a quarter of the resolution (about 10 m), the same way the earlier HEO labels were made. Water is NDWI > 0.05, and no data is where all 7 bands are 0.

Needs the GPU for OmniCloudMask. Nothing is written before step 4. Run it one cell at a time.

## 1 · The products

In [ ]:
# [1] setup
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                # the notebooks run from the repository root
import os, sys, glob, json, importlib
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.transform import array_bounds
from shapely.geometry import box
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import burnscar.labels as L
importlib.reload(L)

PRODUCTS = "data/heo/products"
OUT_IMG, OUT_MSK = "data/heo/images", "data/heo/masks_v3"
INDEX = "data/tables/labels_v3_heo_products.csv"
SCALE = L.reflectance_scale(PRODUCTS)                      # 4094: stored HEO values / SCALE = reflectance
BANDS = ["B", "G", "R", "RE1", "RE2", "RE3", "NIR"]        # product bands B01 to B07, B00 (PAN) is not used
cfg = L.RelabelConfig()                                    # the same v3 rules as notebook 02

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
C_BLUE, C_ORANGE, C_GREY, INK2 = "#2a78d6", "#eb6834", "#8a8984", "#52514e"
SRC_NAME = {"phisat2_sim": "PhiSat-2 sim", "heo": "HEO", "s2_test": "Sentinel-2 test"}
SRC_COLOR = {"phisat2_sim": C_BLUE, "heo": C_ORANGE, "s2_test": C_GREY}
plt.rcParams.update({"axes.grid": True, "grid.color": "#e6e5e0", "axes.axisbelow": True,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
                     "axes.edgecolor": "#8a8984", "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2})
CMAP = ListedColormap([L.V3_COLORS[v] for v in L.V3_VALUES])
NORM = BoundaryNorm([-0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 200, 254.5, 255.5], CMAP.N)

In [ ]:
# [2] the products and their band files
def band_files(prod_dir):
    """{1..7: path} of bands B01 to B07 in the product's IMG_DATA folder."""
    out = {}
    for f in sorted(glob.glob(os.path.join(prod_dir, "**", "*.B0[0-7].tif"), recursive=True), key=len):
        out.setdefault(int(f[-6:-4]), f)
    return {b: out[b] for b in range(1, 8) if b in out}

rows = []
for d in sorted(p for p in glob.glob(f"{PRODUCTS}/IMH*") if os.path.isdir(p)):
    sid = os.path.basename(d)
    bf = band_files(d)
    with rasterio.open(bf[1]) as s:
        rows.append(dict(scene=sid, spacecraft=sid[:5], level=sid[6:10], acq=L.parse_acq_datetime(sid),
                         n_bands=len(bf), width=s.width, height=s.height, px_m=round(s.res[0], 2),
                         crs=str(s.crs), dir=d))
prods = pd.DataFrame(rows).sort_values("acq").reset_index(drop=True)
prods.drop(columns=["dir"])        # expected: 10 products, 7 bands each, 2.5 m

In [ ]:
# [3] product metadata from the STAC json: everything for the first product, then the quality fields of all of them
def stac_properties(prod_dir):
    files = sorted(glob.glob(os.path.join(prod_dir, "**", "*.json"), recursive=True), key=len)
    return json.load(open(files[0])).get("properties", {}) if files else {}

props = {r.scene: stac_properties(r.dir) for r in prods.itertuples()}
display(pd.Series({k: str(v)[:150] for k, v in props[prods.scene[0]].items()}).to_frame(prods.scene[0][:32]))
WORDS = ("coregistration", "gcp", "rms", "cloud", "sun", "view", "lineage", "baseline", "quality")
keys = sorted({k for p in props.values() for k in p if any(w in k.lower() for w in WORDS)})
pd.DataFrame({s[:32]: {k: str(p.get(k, ""))[:60] for k in keys} for s, p in props.items()}).T

## 2 · Quicklooks, and the fires in each product

In [ ]:
# [4] true-colour quicklooks
def rgb_quicklook(bf, decim=8):
    arrs = []
    for b in (3, 2, 1):                                   # red, green, blue
        with rasterio.open(bf[b]) as s:
            arrs.append(s.read(1, out_shape=(s.height // decim, s.width // decim),
                               resampling=Resampling.nearest).astype(np.float32))
    h, w = min(a.shape[0] for a in arrs), min(a.shape[1] for a in arrs)
    a = np.stack([x[:h, :w] for x in arrs])
    valid = (a > 0).all(0)
    out = np.ones((h, w, 3), np.float32)
    for i in range(3):
        lo, hi = np.percentile(a[i][valid], (2, 98)) if valid.any() else (0, 1)
        out[..., i] = np.clip((a[i] - lo) / max(hi - lo, 1e-6), 0, 1)
    out[~valid] = 1
    return out

n = len(prods)
fig, axes = plt.subplots(int(np.ceil(n / 5)), 5, figsize=(20, 4.25 * int(np.ceil(n / 5))), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for ax, r in zip(axes.ravel(), prods.itertuples()):
    ax.imshow(rgb_quicklook(band_files(r.dir)))
    ax.set_title(f"{r.spacecraft} {r.acq:%Y-%m-%d %H:%M:%S}", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# [5] EFFIS, and the fires inside each product by age on the image date
fires, parts = L.load_fires_merged(L.EFFIS_FILES)
print(f"EFFIS: {len(fires):,} polygons, fires from {fires.fire_date.min().date()} to {fires.fire_date.max().date()}")

rows = []
for r in prods.itertuples():
    with rasterio.open(band_files(r.dir)[1]) as s:
        crs, b = s.crs, s.bounds
    sf = L.scene_fires(fires, crs, b, r.acq)
    age = sf.age_days.values
    f, o, _ = L.age_windows(age, cfg.fresh_max, cfg.old_max, cfg.ignore_max)
    ha = sf.geometry.intersection(box(*b)).area.values / 1e4
    rows.append(dict(scene=r.scene[:32], fresh_fires=int(f.sum()), fresh_ha=round(ha[f].sum()),
                     old_fires=int(o.sum()), old_ha=round(ha[o].sum()),
                     older_than_1y_ha=round(ha[age > cfg.old_max].sum()), after_image_ha=round(ha[age < 0].sum())))
fire_tab = pd.DataFrame(rows)
fire_tab       # hectares of burnt-polygon area inside each product's bounding box, before clouds are taken out

## 3 · One product

The functions that make a stack and a mask, then one product to look at. It starts with the product that has the most fresh burnt area; set `SAMPLE` to any product name to see another.

In [ ]:
# [6] stack, clouds, labels, plot
def load_stack(prod_dir):
    """7-band uint16 stack (B, G, R, RE1, RE2, RE3, NIR) on the B01 grid."""
    bf = band_files(prod_dir)
    with rasterio.open(bf[1]) as ref:
        H, W, crs, tr = ref.height, ref.width, ref.crs, ref.transform
    stack = np.zeros((7, H, W), np.uint16)
    for i, b in enumerate(range(1, 8)):
        with rasterio.open(bf[b]) as src:
            if (src.height, src.width) == (H, W) and src.transform == tr:
                stack[i] = src.read(1)
            else:                                          # a band on a slightly different grid
                reproject(rasterio.band(src, 1), stack[i], src_transform=src.transform, src_crs=src.crs,
                          dst_transform=tr, dst_crs=crs, resampling=Resampling.bilinear)
    return stack, crs, tr

def cloud_shadow(stack, valid, device="cuda:0", f=4):
    """OmniCloudMask on red, green and NIR at 1/f of the resolution, as for the earlier HEO labels."""
    import cv2
    from omnicloudmask import predict_from_array
    r, g, nir = (stack[k].astype(np.float32) for k in (2, 1, 6))
    small = np.stack([cv2.resize(x, (x.shape[1] // f, x.shape[0] // f), interpolation=cv2.INTER_AREA)
                      for x in (r, g, nir)])
    pred = np.asarray(predict_from_array(small, batch_size=1, apply_no_data_mask=True, no_data_value=0,
                                         inference_device=device))[0]
    pred = cv2.resize(pred.astype(np.uint8), (r.shape[1], r.shape[0]), interpolation=cv2.INTER_NEAREST)
    return ((pred == 1) | (pred == 2)) & valid, (pred == 3) & valid      # thick or thin cloud, shadow

def label_scene(prod_dir, acq):
    stack, crs, tr = load_stack(prod_dir)
    H, W = stack.shape[1:]
    valid = ~np.all(stack == 0, axis=0)
    cloud, shadow = cloud_shadow(stack, valid)
    water = L.ndwi_water(stack[1], stack[6], cfg.ndwi_threshold) & valid
    base = np.full((H, W), L.CLEAR, np.uint8)             # background first, compose_v3 paints the burns
    base[water] = L.WATER
    base[shadow] = L.SHADOW
    base[cloud] = L.CLOUD
    base[~valid] = L.NODATA
    sf = L.scene_fires(fires, crs, array_bounds(H, W, tr), acq)
    f, o, i = L.age_windows(sf.age_days.values, cfg.fresh_max, cfg.old_max, cfg.ignore_max)
    edge_px = int(round(cfg.edge_m / abs(tr.a))) if cfg.edge_m > 0 else 0
    mask = L.compose_v3(base, L.rasterize_bool(sf.geometry[f], (H, W), tr),
                        L.rasterize_bool(sf.geometry[o], (H, W), tr),
                        L.rasterize_bool(sf.geometry[i], (H, W), tr), edge_px=edge_px)
    refl = [stack[k].astype(np.float32) / SCALE for k in (0, 1, 2, 6)]          # B, G, R, NIR
    mask = L.clean_burns(mask, *refl, ndvi_max=cfg.ndvi_max, bright_max=cfg.bright_max)
    return stack, mask, crs, tr

def show_scene(stack, mask, title, decim=4):
    a = stack[[6, 2, 1], ::decim, ::decim].astype(np.float32)          # NIR, red, green
    valid = (a > 0).all(0)
    img = np.ones(a.shape[1:] + (3,), np.float32)
    for k in range(3):
        lo, hi = np.percentile(a[k][valid], (2, 98)) if valid.any() else (0, 1)
        img[..., k] = np.clip((a[k] - lo) / max(hi - lo, 1e-6), 0, 1)
    img[~valid] = 1
    fig, ax = plt.subplots(1, 2, figsize=(15, 7.5))
    ax[0].imshow(img); ax[0].set_title(f"{title}   NIR, red, green", loc="left", fontsize=10)
    ax[1].imshow(mask[::decim, ::decim], cmap=CMAP, norm=NORM, interpolation="nearest")
    ax[1].set_title("labels v3", loc="left", fontsize=10)
    for x in ax:
        x.axis("off")
    present = sorted(np.unique(mask).tolist())
    fig.legend(handles=[Patch(facecolor=L.V3_COLORS[v], edgecolor="#8a8984", label=L.V3_NAMES[v]) for v in present],
               loc="lower center", ncol=len(present))
    plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

In [ ]:
# [7] one product
SAMPLE = prods.scene[fire_tab.fresh_ha.idxmax()]
r = prods.set_index("scene").loc[SAMPLE]
stack, mask, crs, tr = label_scene(r.dir, r.acq)
print(SAMPLE)
print({L.V3_NAMES[int(k)]: int(n) for k, n in zip(*np.unique(mask, return_counts=True))})
show_scene(stack, mask, SAMPLE[:32])

## 4 · Label and write all products

A few minutes. Writes a stack and a mask per product (overwriting what this notebook wrote before), and the class counts to `data/tables/labels_v3_heo_products.csv`.

In [ ]:
# [8] label and write
os.makedirs(OUT_IMG, exist_ok=True); os.makedirs(OUT_MSK, exist_ok=True); os.makedirs("data/tables", exist_ok=True)
rows = []
for r in prods.itertuples():
    stack, mask, crs, tr = label_scene(r.dir, r.acq)
    prof = dict(driver="GTiff", height=stack.shape[1], width=stack.shape[2], crs=crs, transform=tr,
                compress="deflate", tiled=True, blockxsize=512, blockysize=512)
    with rasterio.open(f"{OUT_IMG}/{r.scene}_img.tif", "w", count=7, dtype="uint16", nodata=0,
                       predictor=2, **prof) as dst:
        dst.write(stack)
        for k, name in enumerate(BANDS, 1):
            dst.set_band_description(k, name)
    L.write_mask(f"{OUT_MSK}/{r.scene}_mask.tif", mask, prof, cfg)
    counts = L.class_counts(mask, "v3_")
    rows.append(dict(scene=r.scene, source="heo", status="ok", acq=r.acq, px_m=abs(tr.a), **counts))
    print(f"{r.scene[:32]}  fresh {counts['v3_fresh']:>9,}  old {counts['v3_old']:>9,}  "
          f"cloud {counts['v3_cloud']:>10,}  ignore {counts['v3_ignore']:>8,}")
pd.DataFrame(rows).to_csv(INDEX, index=False)
with open(INDEX.replace(".csv", "_config.json"), "w") as f:
    json.dump({**asdict(cfg), "effis": L.EFFIS_FILES, "n_products": len(rows),
               "clouds": "OmniCloudMask, red/green/NIR at 1/4 resolution", "created": pd.Timestamp.now().isoformat(),
               "legend": {str(k): v for k, v in L.V3_NAMES.items()}}, f, indent=2)
print(len(rows), "products written")

## 5 · What the products add

In [ ]:
# [9] HEO class balance: the 25 earlier scenes (notebook 02) next to the products
prod_df = pd.read_csv(INDEX)
cols = [f"v3_{n}" for n in L.V3_NAMES.values()]

def share(d):
    s = d[cols].sum().drop("v3_nodata")
    return 100 * s / s.sum()

groups = {"products": prod_df}
if os.path.exists("data/tables/labels_v3_from_v2.csv"):
    earlier = pd.read_csv("data/tables/labels_v3_from_v2.csv").query("source == 'heo'")
    groups = {"25 earlier scenes": earlier, "products": prod_df,
              "all HEO": pd.concat([earlier, prod_df], ignore_index=True)}
else:
    print("run notebook 02 with WRITE = True to compare with the 25 earlier HEO scenes")
bal = pd.DataFrame({k: share(v) for k, v in groups.items()})
bal.index = [i[3:] for i in bal.index]
display(bal.round(3))
for n in ["fresh", "old"]:
    print(f"{n:<6} pixels: " + " | ".join(f"{k} {int(v[f'v3_{n}'].sum()):,}" for k, v in groups.items()))

fig, ax = plt.subplots(figsize=(11, 4))
x, w = np.arange(len(bal)), 0.8 / len(bal.columns)
for i, (col, color) in enumerate(zip(bal.columns, [C_GREY, C_ORANGE, "#52514e"])):
    bars = ax.bar(x + (i - (len(bal.columns) - 1) / 2) * w, bal[col].clip(lower=1e-4), w * 0.92, color=color, label=col)
    ax.bar_label(bars, labels=[f"{v:.2g}" for v in bal[col]], fontsize=7, color=INK2, padding=2)
ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(bal.index)
ax.set_ylabel("% of pixels with data (log scale)")
ax.set_title("HEO class balance", loc="left", fontsize=10)
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

In [ ]:
# [10] burnt pixels per product, to choose the HEO scenes we keep for testing.
# IMH06 20260821T123054 and 20260821T123102 are from the same pass, they have to go in the same split.
t = (prod_df.assign(scene=prod_df.scene.str[:32])[["scene", "v3_fresh", "v3_old", "v3_ignore", "v3_cloud", "v3_nodata"]]
            .sort_values("v3_fresh", ascending=False).reset_index(drop=True))
display(t)
fig, ax = plt.subplots(figsize=(10, 0.45 * len(t) + 1))
y = np.arange(len(t))
ax.barh(y - 0.2, t.v3_fresh, 0.4, color=L.V3_COLORS[L.FRESH], label="fresh burn")
ax.barh(y + 0.2, t.v3_old, 0.4, color=L.V3_COLORS[L.OLD], label="old burn")
ax.set_yticks(y); ax.set_yticklabels(t.scene, fontsize=8); ax.invert_yaxis()
ax.set_xlabel("pixels (2.5 m)"); ax.legend(loc="lower right")
ax.set_title("burnt pixels per product", loc="left", fontsize=10)
plt.tight_layout(); plt.show()